In [ ]:
repository_filter: list[str] = []

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

# df = read_data_table("../samples/test_quality_issues.csv")
df = read_data_table("../samples/v2/io.moderne.prethink.table.TestQualityIssues.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    severity_color_map = {
        "low": qu.SEVERITY_COLORS["LOW"],
        "medium": qu.SEVERITY_COLORS["MEDIUM"],
        "high": qu.SEVERITY_COLORS["HIGH"],
    }
    severity_order = ["low", "medium", "high"]

    grouped = df.groupby(["issueType", "severity"]).size().reset_index(name="count")

    fig = go.Figure()

    for sev in severity_order:
        subset = grouped[grouped["severity"] == sev]
        if len(subset) == 0:
            continue
        fig.add_trace(
            go.Scatter(
                x=subset["issueType"],
                y=subset["severity"],
                mode="markers+text",
                marker=dict(
                    size=subset["count"] * 5 + 15,
                    color=severity_color_map.get(sev, "#999"),
                    line=dict(width=1, color="#666"),
                ),
                text=subset["count"].astype(str),
                textposition="middle center",
                textfont=dict(size=11, color="#333"),
                name=sev,
                hovertemplate=(
                    "<b>%{x}</b><br>"
                    "Severity: %{y}<br>"
                    "Count: %{text}"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title="Test Quality Risk Matrix",
        xaxis_title="Issue Type",
        yaxis_title="Severity",
        yaxis=dict(categoryorder="array", categoryarray=severity_order),
        height=500,
        margin=dict(l=100, r=50, t=60, b=120),
        showlegend=True,
        xaxis=dict(tickangle=-45),
    )

fig.show()